# 03 - Churn Model & Retention Strategy
Loads the segmented customer table from notebook 02, trains the churn model, builds the retention matrix and exports the final dataset.

**Input:** `customer_segments.csv`  
**Output:** `olist_customer_retention_export.csv`

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [ ]:
# Mount Google Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/olist/'          # <-- raw Olist CSVs (update to your folder)
processed_path = drive_path + 'processed/'            # <-- intermediate files shared between notebooks
os.makedirs(processed_path, exist_ok=True)

In [ ]:
# ==============================================================================
# LOAD SEGMENTED DATA (output of 02_eda_and_segmentation.ipynb)
# ==============================================================================
customer_features = pd.read_csv(processed_path + 'customer_segments.csv', parse_dates=['last_order'])
print(f"customer_features: {customer_features.shape[0]} customers, {customer_features.shape[1]} columns")

## Churn model

In [ ]:
# ==============================================================================
# STEP 6: CHURN MODEL (Supervised Logistic Regression)
# ==============================================================================

# 1. Compare average behavior: Retained (0) vs Churned (1)
# Evaluates baseline behavioral differences between active and churned customers
driver_cols = ["late_order_rate", "average_review_score", "average_freight_cost", "frequency"]
churn_compare = customer_features.groupby("churn_risk")[driver_cols].mean()
churn_compare.T.plot(kind="bar", figsize=(9, 4), color=["#10B981", "#D9534F"])
plt.title("Average Behavior: Retained (0) vs Churned (1)")
plt.ylabel("Mean Value")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# 2. Train-test split with stratified sampling
feature_cols = [
    "frequency", "monetary_value", "average_order_value", "category_count",
    "average_review_score", "late_order_rate", "average_freight_cost", "average_installments"
]

model_df = customer_features.dropna(subset=feature_cols + ["churn_risk"])
X = model_df[feature_cols]
y = model_df["churn_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 3. Standardize features and train Logistic Regression
scaler_model = StandardScaler()
X_train_scaled = scaler_model.fit_transform(X_train)
X_test_scaled = scaler_model.transform(X_test)

logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg.fit(X_train_scaled, y_train)

# 4. Predict and evaluate performance metrics
y_pred = logreg.predict(X_test_scaled)
y_prob = logreg.predict_proba(X_test_scaled)[:, 1]

print("--- Model Performance ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}\n")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 5. Extract and plot standardized feature coefficients (Churn Drivers)
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logreg.coef_[0]
}).sort_values("coefficient")

plt.figure(figsize=(9, 4))
plt.barh(
    coef_df["feature"],
    coef_df["coefficient"],
    color=np.where(coef_df["coefficient"] > 0,"#10B981", "#D9534F")
)
plt.axvline(0, color="black", linestyle="--", linewidth=0.8)
plt.title("Churn Drivers (Standardized Logistic Regression Coefficients)")
plt.xlabel("Coefficient (Impact on Churn)")
plt.tight_layout()
plt.show()

# 6. Score individual churn probability back into the master table
customer_features.loc[model_df.index, "churn_probability"] = logreg.predict_proba(
    scaler_model.transform(model_df[feature_cols])
)[:, 1]

## Retention strategy & export

In [ ]:
# ==============================================================================
# STEP 7: BUSINESS STRATEGY, ACTIONABLE RETENTION MATRIX & EXPORT
# ==============================================================================

# 1. Map individual churn probabilities and segments into actionable retention strategies
def assign_action(row):
    prob = row["churn_probability"]
    segment = row["segment"]

    if segment in ["Champions", "Loyal customers"]:
        if prob >= 0.70:
            return "Priority VIP Recovery (Direct Outreach & Perks)"
        elif prob >= 0.40:
            return "Proactive Retention (Loyalty Rewards)"
        else:
            return "Upsell & Advocacy (Early Access / Cross-sell)"
    elif segment == "At-risk customers":
        if row["late_order_rate"] > 0:
            return "Logistics Service Recovery (Shipping Voucher)"
        else:
            return "Win-Back Discount Campaign"
    else:  # Lost customers
        return "Automated Drip Email / Low-Cost Reactivation"

customer_features["retention_strategy"] = customer_features.apply(assign_action, axis=1)

# 2. Quantify financial exposure (Revenue at Risk per Segment)
# Evaluates financial exposure: quantifies total revenue at high risk of churn per segment

risk_summary = customer_features.groupby("segment").agg(
    total_customers=("customer_unique_id", "count"),
    avg_churn_prob=("churn_probability", "mean"),
    total_revenue=("monetary_value", "sum"),
    revenue_at_risk=("monetary_value", lambda x: x[customer_features.loc[x.index, "churn_probability"] >= 0.50].sum())
).reset_index()

print("--- Segment Financial Exposure Summary ---")
display(risk_summary)
# 3. Visualize Total Revenue vs. Revenue at Risk
# Visualizes which customer segments carry the largest monetary churn exposure

fig, ax = plt.subplots(figsize=(9, 4.5))
risk_summary.plot(
    x="segment",
    y=["total_revenue", "revenue_at_risk"],
    kind="bar",
    color=["#2E86AB", "#D9534F"],
    edgecolor="white",
    linewidth=1.2,
    width=0.65,
    ax=ax
)
plt.title("Total Revenue vs. Revenue at Risk (>50% Churn Probability)")
plt.ylabel("Monetary Value (BRL)")
plt.xlabel("RFM Segment")
plt.xticks(rotation=15)
plt.legend(["Total Revenue", "Revenue at Risk"], frameon=True, facecolor="white")
plt.tight_layout()
plt.show()

# 4. Filter Priority Intervention Targets (High-value spenders slipping away)
# Identifies top high-value VIP accounts in active danger of leaving for immediate customer service outreach

priority_vips = customer_features[
    (customer_features["segment"].isin(["Champions", "Loyal customers"])) &
    (customer_features["churn_probability"] >= 0.60)
].sort_values(by=["monetary_value", "churn_probability"], ascending=[False, False])

print(f"\nPriority High-Value Customers for Immediate Intervention: {len(priority_vips)}")
display(priority_vips[[
    "customer_unique_id", "segment", "monetary_value",
    "churn_probability", "average_review_score", "late_order_rate"
]].head(10))

# 5. Export master dataset for Business Intelligence (Power BI, Tableau, Streamlit)
# "Exports unified customer analytics table ready for executive dashboards.")
export_cols = [
    "customer_unique_id", "recency_days", "frequency", "monetary_value",
    "average_order_value", "average_review_score", "late_order_rate",
    "segment", "cluster", "churn_risk", "churn_probability", "retention_strategy"
]
customer_features[export_cols].to_csv(processed_path + "olist_customer_retention_export.csv", index=False)
print("\nDataset successfully saved to 'processed/olist_customer_retention_export.csv'!")